# 한국마사회 구매성향분석 (2022-01 ~ 2026-07)

한국마사회 공공데이터포털 OpenAPI(`pchTndcyAnal`, https://www.data.go.kr/data/15154736/openapi.do )에서 월별·시행경마장·판매경마장 조합으로 수집한 마권 구매성향 데이터를 분석합니다.

- 데이터: `raw/한국마사회_구매성향분석_202201-202607.csv` (55개월 × 3개 시행경마장 × 3개 판매경마장 = 495행)
- 컬럼: `performRacecourse`(시행경마장, 경주가 열리는 곳) · `saleRacecourse`(판매경마장, 마권이 팔린 채널) · `raceMonth` · `saleCount`(발매건수) · `saleValue`(발매금액, 원)

> 이 API는 (경주년월 × 시행경마장 × 판매경마장) 조합 하나당 결과 1건만 반환하는 구조라, 여러 조합을 반복 호출해 CSV로 모았습니다. `04:양천`은 실제 경주가 열리지 않는 장외발매소라 조합에서 제외했습니다(호출 시 0건).

**요약 결론**: 경주 개최 비중은 서울 49.5%·부산경남 29.2%·제주 21.3%로 비교적 고르게 분산되어 있지만, 실제 마권 판매는 **판매경마장(채널) 기준 서울이 전체의 91.5%를 독점**합니다. 게다가 이 쏠림은 완화되지 않고 **2022년 91.2% → 2026년 93.2%로 오히려 매년 심화**되고 있습니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['axes.unicode_minus'] = False
# plt.rc('font', family='NanumGothic')  # 한글 폰트가 있다면 주석 해제 (Linux/Windows)
# plt.rc('font', family='AppleGothic')  # macOS

df = pd.read_csv('raw/한국마사회_구매성향분석_202201-202607.csv')
df['year'] = df['raceMonth'] // 100
df['month'] = df['raceMonth'] % 100
df['is_self'] = df['performRacecourse'] == df['saleRacecourse']

print(df.shape)
df.head()

## 1. 데이터 확인

결측치, 기간, 조합 개수를 확인합니다.

In [ ]:
print("결측치:")
print(df.isnull().sum())
print()
print(f"기간: {df['raceMonth'].min()} ~ {df['raceMonth'].max()} ({df['raceMonth'].nunique()}개월)")
print(f"중복 조합: {df.duplicated(subset=['performRacecourse','raceMonth','saleRacecourse']).sum()}건")
print(f"시행경마장: {sorted(df['performRacecourse'].unique())}")
print(f"판매경마장: {sorted(df['saleRacecourse'].unique())}")
print(f"전체 발매금액: {df['saleValue'].sum():,}원 ({df['saleValue'].sum()/1e12:.2f}조원)")
print(f"전체 발매건수: {df['saleCount'].sum():,}건")

## 2. 월별 추이

2022~2025년은 온전한 연도, 2026년은 1~7월만 존재합니다. 연도 비교는 월평균 기준으로 봅니다.

In [ ]:
monthly = df.groupby('raceMonth')['saleValue'].sum()

fig, ax = plt.subplots(figsize=(11, 4))
monthly.plot(ax=ax, color='#12897B', marker='o', markersize=2.5)
ax.set_title('월별 총 발매금액 추이')
ax.set_ylabel('발매금액(원)')
plt.tight_layout()
plt.show()

year_avg = df.groupby('year').apply(lambda g: g.groupby('raceMonth')['saleValue'].sum().mean())
print("연도별 월평균 발매금액(억원):")
print((year_avg / 1e8).round(1))
print()
for y in range(2023, 2027):
    if y in year_avg.index and (y - 1) in year_avg.index:
        print(f"{y} vs {y-1} YoY(월평균): {(year_avg[y]/year_avg[y-1]-1)*100:+.1f}%")

**인사이트**: 2022~2026년(1~7월) 동안 월평균 발매금액은 5,330억~5,505억원 사이에서 등락하며 뚜렷한 추세 없이 안정적입니다(연도별 증감 -1.4%~+2.8%). 규모 자체보다 아래에서 보는 **채널 쏠림**이 이 데이터의 핵심입니다.

## 3. 계절성

In [ ]:
season = df.groupby('month')['saleValue'].sum()

fig, ax = plt.subplots(figsize=(8, 4))
top3 = set(season.sort_values(ascending=False).index[:3])
colors = ['#0B7A69' if m in top3 else '#AEE0D3' for m in season.index]
season.plot.bar(ax=ax, color=colors)
ax.set_title('월별(계절) 발매금액 합산 (2022~2026 누적)')
ax.set_xlabel('월')
plt.tight_layout()
plt.show()

**인사이트**: 3·5·7월이 가장 높고, 9월이 가장 낮습니다 — 봄~초여름 경마 성수기와 가을 비수기가 뚜렷합니다.

## 4. 시행경마장 vs 판매경마장 — 채널 쏠림

⚠️ 이 데이터셋의 핵심 발견입니다. 경주가 실제로 열리는 곳(시행경마장)과 마권이 팔리는 채널(판매경마장)을 분리해서 봅니다.

In [ ]:
perf_share = df.groupby('performRacecourse')['saleValue'].sum().sort_values(ascending=False)
sale_share = df.groupby('saleRacecourse')['saleValue'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
(perf_share / perf_share.sum() * 100).plot.bar(ax=axes[0], color='#12897B')
axes[0].set_title('시행경마장별 경주 비중 (%)')
axes[0].set_ylabel('%')

(sale_share / sale_share.sum() * 100).plot.bar(ax=axes[1], color='#C2501F')
axes[1].set_title('판매경마장(채널)별 발매 비중 (%)')
axes[1].set_ylabel('%')

plt.tight_layout()
plt.show()

print("시행경마장별 비중(%):")
print((perf_share / perf_share.sum() * 100).round(1))
print("\n판매경마장(채널)별 비중(%):")
print((sale_share / sale_share.sum() * 100).round(1))

**인사이트**: 경주 개최 비중은 서울 49.5%·부산경남 29.2%·제주 21.3%로 비교적 고르지만, 판매 채널은 **'서울' 채널이 전체 발매금액의 91.5%를 독점**합니다. 부산경남·제주 채널은 각각 4.2%·4.3%에 불과합니다.

### 4.1 경마장별 자체판매 비중

In [ ]:
self_cross = df.groupby(['performRacecourse', 'is_self'])['saleValue'].sum().unstack()
self_cross.columns = ['타채널 판매', '자체채널 판매']
self_cross['자체채널 비중(%)'] = (self_cross['자체채널 판매'] / (self_cross['자체채널 판매'] + self_cross['타채널 판매']) * 100).round(1)
self_cross

**인사이트**: 서울 경주는 92.4%가 서울 채널에서 자체 판매되는 반면, **부산경남 경주는 95.3%, 제주 경주는 92.3%가 '서울' 채널을 통해 팔립니다**. 경주 개최지와 마권 판매 채널이 사실상 분리되어 있습니다.

### 4.2 서울 채널 집중도 추이 — 완화되지 않는 쏠림

연도별로 '서울' 채널 비중이 어떻게 변해왔는지 봅니다.

In [ ]:
yearly_seoul_share = df.groupby('year').apply(lambda g: g[g['saleRacecourse'] == '서울']['saleValue'].sum() / g['saleValue'].sum() * 100)

fig, ax = plt.subplots(figsize=(7, 4))
yearly_seoul_share.plot(ax=ax, marker='o', color='#C2501F')
ax.set_title("연도별 '서울' 판매채널 비중 추이")
ax.set_ylabel('%')
ax.set_ylim(88, 95)
plt.tight_layout()
plt.show()

yearly_seoul_share.round(1)

**인사이트 (신규 발견)**: 서울 채널 비중은 2022년 91.2%에서 2026년(1~7월) 93.2%로 **매년 꾸준히 상승**하고 있습니다. 시간이 지날수록 지역 채널의 입지가 오히려 더 좁아지는 추세입니다 — 6절 액션 플랜의 우선순위를 뒷받침하는 근거입니다.

## 5. 채널별 구매 행태 — 제주 채널의 큰손

In [ ]:
avg_bet = df.groupby('saleRacecourse').apply(lambda g: g['saleValue'].sum() / g['saleCount'].sum())
avg_bet = avg_bet.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
avg_bet.plot.bar(ax=ax, color='#12897B')
ax.set_title('판매경마장(채널)별 평균 구매단가')
ax.set_ylabel('원/건')
plt.tight_layout()
plt.show()

avg_bet.round(0)

**인사이트**: 제주 채널 이용자의 평균 구매단가(15,752원/건)가 서울(9,498원)·부산경남(8,062원)보다 뚜렷하게 높습니다. 판매 건수는 적지만 건당 구매 규모가 큰 채널입니다.

## 6. 개선 액션 플랜

### 즉시 실행 (0-3개월)
1. **지역 판매 채널 활성화 진단** — 부산경남·제주 경주의 95%·92%가 서울 채널로 판매되는 원인(오프라인 발매소 접근성, 온라인 채널 지역 배정 방식 등)을 진단합니다. *(근거: 부산경남 자체채널 비중 4.7%, 제주 7.7%)*
2. **서울 채널 부하 모니터링** — 전체 발매금액의 91.5%가 몰리는 서울 채널의 시스템·발매소 처리 용량을 점검합니다.

### 중기 과제 (3-12개월)
1. **채널별 맞춤 프로모션** — 제주 채널처럼 건당 구매단가가 높은 채널은 고액 구매자 대상 서비스를, 부산경남처럼 자체 채널 비중이 낮은 지역은 접근성 개선을 검토합니다.
2. **계절 성수기 대비 인력·시스템 배치** — 3·5·7월 성수기 직전에 발매 시스템·인력을 선제적으로 확충합니다.

### 장기 과제 (1년 이상)
1. **심화되는 채널 쏠림 추세 대응** — 서울 채널 집중도가 2022년 91.2%에서 2026년 93.2%로 매년 상승하고 있습니다. 방치할 경우 지역 경마장의 판매 기반이 더 약화될 수 있어, 지역 채널 강화를 위한 중장기 로드맵이 필요합니다. *(근거: 5년간 서울 채널 비중 +2.0%p 상승)*

## 7. 데이터 한계 및 방법론 노트

- 이 데이터는 시행경마장 3곳(서울·제주·부산경남) × 판매경마장 3곳의 광역 조합만 존재합니다(04:양천은 장외발매소라 경주가 열리지 않아 조합에서 제외). 개별 발매소 단위의 지역 분포는 이 데이터로 알 수 없습니다.
- 2026년은 1~7월 데이터만 수집했습니다(API 호출 시점 기준). 연도 비교는 총합이 아닌 월평균 기준으로 계산했습니다.
- '취소건수', '환불금액' 등 공공데이터포털 설명에 언급된 필드는 이 API 응답에는 포함되어 있지 않았습니다(발매건수·발매금액만 제공).
- 본 분석은 서술 통계 기반이며, 제시된 관계는 상관관계로 인과관계를 증명하지 않습니다.